# 09c · Learning in the C monad: the update arithmetic, in C (paper §7)
**Requires the C kernel.** The lines below are extracted **verbatim** from `VAPMIP/PtolC/monad.c` at build time and run on a minimal stand-in for the `Monad` struct (only the fields those lines touch); constants come from the real `ptolemy.h` and `filter.h`:
`dirac_pole` (lines 443–446), the β update (491–494), the `prose_seen` ladder (505–512), the co-occurrence coupling (556–561) of `monad_learn_ex`.

In [1]:
//%cflags:-I../../PtolC -D_DEFAULT_SOURCE
//%ldflags:-lm
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <string.h>
#include "ptolemy.h"
#include "filter.h"

/* monad.c:443-446 */
static int dirac_pole(double gam)
{
    return (sin(gam * 0.5) * cos(gam * 0.5) > 0.0) ? 1 : -1;
}

typedef struct { double zeros[4]; double beta[4]; struct { int prose_seen; } vocab[4]; } MiniMonad;

/* monad.c:491-494, verbatim, on slot idx of a MiniMonad */
static void beta_update(MiniMonad *m, int idx, double E) {
            double amp = fabs(sin(m->zeros[idx])) * (M_PI * 0.5);
            double nb = m->beta[idx] + E * E * MONAD_ALPHA_LEARN * amp;
            if (nb > MONAD_BETA_SAT) nb = MONAD_BETA_SAT;
            m->beta[idx] = nb;
}

int main(void) {
    MiniMonad m = { .zeros = {14.134725, 21.022040, 25.010858, 30.424876}, .beta = {0, 0, 0, 0} };
    double E = 0.40;
    printf("MONAD_ALPHA_LEARN = %g, MONAD_BETA_SAT = %g\n", MONAD_ALPHA_LEARN, MONAD_BETA_SAT);
    int monotone = 1, capped = 1; double prev = 0;
    for (int step = 1; step <= 4000; step++) {
        beta_update(&m, 0, E);
        if (m.beta[0] + 1e-15 < prev) monotone = 0;
        if (m.beta[0] > MONAD_BETA_SAT + 1e-12) capped = 0;
        prev = m.beta[0];
        if (step == 1 || step == 10 || step == 100 || step == 1000 || step == 4000) printf("step %4d: beta = %.6f\n", step, m.beta[0]);
    }
    printf("%s: beta only rises with repeated exposure and never exceeds MONAD_BETA_SAT\n", (monotone && capped) ? "PASS" : "FAIL");
    return 0;
}


MONAD_ALPHA_LEARN = 0.01, MONAD_BETA_SAT = 7.552
step    1: beta = 0.002513
step   10: beta = 0.025133
step  100: beta = 0.251327
step 1000: beta = 2.513267


step 4000: beta = 7.552000
PASS: beta only rises with repeated exposure and never exceeds MONAD_BETA_SAT


## The `prose_seen` ladder is order-dependent

In [2]:
//%cflags:-I../../PtolC -D_DEFAULT_SOURCE
//%ldflags:-lm
#include <stdio.h>
#include "filter.h"

typedef struct { struct { int prose_seen; } vocab[1]; } MiniMonad;

/* monad.c:505-512, verbatim, on slot idx of a MiniMonad */
static void ladder(MiniMonad *m, int idx, NSFiletype ft) {
            if (ft == NS_FT_WORDNET)
                m->vocab[idx].prose_seen = 2;          /* canonical dictionary */
            else if (ft == NS_FT_PROSE || ft == NS_FT_MARKUP || ft == NS_FT_DOC) {
                if (m->vocab[idx].prose_seen == 0)
                    m->vocab[idx].prose_seen = 1;       /* prose only */
                else if (m->vocab[idx].prose_seen == 2)
                    m->vocab[idx].prose_seen = 3;       /* WN + prose = verified common */
            }
}

int main(void) {
    MiniMonad a = { .vocab = { {0} } }, b = { .vocab = { {0} } };
    ladder(&a, 0, NS_FT_WORDNET); ladder(&a, 0, NS_FT_PROSE);      /* dictionary first, then prose */
    ladder(&b, 0, NS_FT_PROSE);   ladder(&b, 0, NS_FT_WORDNET);    /* prose first, then dictionary */
    printf("WordNet then prose: prose_seen = %d\nprose then WordNet: prose_seen = %d\n", a.vocab[0].prose_seen, b.vocab[0].prose_seen);
    printf("%s: the ladder reaches 3 (verified common) in one order and not the other (%d vs %d)\n", (a.vocab[0].prose_seen != b.vocab[0].prose_seen) ? "PASS" : "FAIL", a.vocab[0].prose_seen, b.vocab[0].prose_seen);
    return 0;
}


WordNet then prose: prose_seen = 3
prose then WordNet: prose_seen = 2
PASS: the ladder reaches 3 (verified common) in one order and not the other (3 vs 2)


## The co-occurrence coupling

In [3]:
//%cflags:-I../../PtolC -D_DEFAULT_SOURCE
//%ldflags:-lm
#include <stdio.h>
#include <math.h>
#include "ptolemy.h"

/* monad.c:443-446 */
static int dirac_pole(double gam)
{
    return (sin(gam * 0.5) * cos(gam * 0.5) > 0.0) ? 1 : -1;
}

typedef struct { double zeros[4]; } MiniMonad;

/* monad.c:556-561, verbatim, for tokens i and j of one sentence */
static double coupling(MiniMonad *m, int *sidx, double *sE, int i, int j) {
                double d_zero = fabs(m->zeros[sidx[i]] - m->zeros[sidx[j]])
                                + MONAD_GAP;
                double d_text = (double)(j - i);
                double couple = (dirac_pole(m->zeros[sidx[i]])
                                 == dirac_pole(m->zeros[sidx[j]])) ? 2.0 : 1.0;
                double w      = sE[i] * sE[j] * couple / (d_zero * d_text);
    return w;
}

int main(void) {
    MiniMonad m = { .zeros = {14.134725, 21.022040, 25.010858, 30.424876} };
    int sidx[4] = {0, 1, 2, 3}; double sE[4] = {0.40, 0.41, 0.42, 0.43};
    int ok = 1;
    for (int j = 1; j < 4; j++) { double w = coupling(&m, sidx, sE, 0, j); printf("token 0 -- token %d: w = %.6e\n", j, w); if (!(w > 0)) ok = 0; }
    double near = coupling(&m, sidx, sE, 0, 1), far = coupling(&m, sidx, sE, 0, 3);
    printf("%s: couplings are positive and the nearer token (distance 1) couples more strongly than the farther (distance 3)\n", (ok && near > far) ? "PASS" : "FAIL");
    return 0;
}


token 0 -- token 1: w = 4.761890e-02
token 0 -- token 2: w = 7.722831e-03
token 0 -- token 3: w = 3.519356e-03
PASS: couplings are positive and the nearer token (distance 1) couples more strongly than the farther (distance 3)
